In [1]:
import os

import torch


# import sys
# sclembas = '/home/hmbaghda/Projects/scLEMBAS'
# sys.path.insert(1, os.path.join(sclembas))
# from scLEMBAS import io
import sys, os
cpa_path = '/home/hmbaghda/Projects/scLEMBAS/notebooks/cpa'
sys.path.insert(1, os.path.join(cpa_path))
from cpa import CPA
from cpa._data import AnnDataSplitter
from cpa._task import CPATrainingPlan
from scvi.dataloaders import DataSplitter
from tqdm import tqdm
from pytorch_lightning.callbacks import EarlyStopping
from scvi.train._callbacks import SaveBestState

[rank: 0] Global seed set to 0


In [2]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

In [3]:
# https://cpa-tools.readthedocs.io/en/latest/tutorials/combosciplex.html#Data-Loading

import scanpy as sc
adata = sc.read(os.path.join(data_path, 'raw', 'combo_sciplex_prep_hvg_filtered.h5ad'))
adata.X = adata.layers['counts'].copy()


In [5]:
import random
random.seed(seed)
adata.obs['cell_types_rand'] = random.choices(['macrophages', 'T_cells', 'B_cells'], k=adata.obs.shape[0])
adata.obs['cat_2'] = random.choices(['A', 'B', 'C', 'D'], k=adata.obs.shape[0])

In [4]:
# import random
# random.seed(seed)
# subset_barcodes = random.sample(population = adata.obs.index.tolist(), k = round(adata.shape[0]*0.01))

# adata = adata[subset_barcodes, :]

In [6]:
CPA.setup_anndata(adata,
                      perturbation_key='condition_ID',
                      dosage_key='log_dose',
                      control_group='CHEMBL504',
                      batch_key=None,
                      is_count_data=True,
                      categorical_covariate_keys=['cell_types_rand', 'cat_2'],
                      deg_uns_key='rank_genes_groups_cov',
                      deg_uns_cat_key='cov_drug_dose',
                      max_comb_len=2,
                     )

ae_hparams = {
    "n_latent": 128,
    "recon_loss": "nb",
    "doser_type": "logsigm",
    "n_hidden_encoder": 512,
    "n_layers_encoder": 3,
    "n_hidden_decoder": 512,
    "n_layers_decoder": 3,
    "use_batch_norm_encoder": True,
    "use_layer_norm_encoder": False,
    "use_batch_norm_decoder": True,
    "use_layer_norm_decoder": False,
    "dropout_rate_encoder": 0.1,
    "dropout_rate_decoder": 0.1,
    "variational": False,
    "seed": 434,
}

trainer_params = {
    "n_epochs_kl_warmup": None,
    "n_epochs_pretrain_ae": 30,
    "n_epochs_adv_warmup": 50,
    "n_epochs_mixup_warmup": 3,
    "mixup_alpha": 0.1,
    "adv_steps": 2,
    "n_hidden_adv": 64,
    "n_layers_adv": 2,
    "use_batch_norm_adv": True,
    "use_layer_norm_adv": False,
    "dropout_rate_adv": 0.3,
    "reg_adv": 20.0,
    "pen_adv": 20.0,
    "lr": 0.0003,
    "wd": 4e-07,
    "adv_lr": 0.0003,
    "adv_wd": 4e-07,
    "adv_loss": "cce",
    "doser_lr": 0.0003,
    "doser_wd": 4e-07,
    "do_clip_grad": False,
    "gradient_clip_value": 1.0,
    "step_size_lr": 45,
}


self = CPA(adata=adata,
                split_key='split_1ct_MEC',
                train_split='train',
                valid_split='valid',
                test_split='ood',
                **ae_hparams,
               )


100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 893.12it/s]
2024-03-26 10:46:55.515950: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


[rank: 0] Global seed set to 434


In [ ]:
# module = CPAModule(
#             n_genes=adata.n_vars,
#             n_perts=len(self.pert_encoder),
#             covars_encoder=self.covars_encoder,
#             **hyper_params,
#         ).float()

In [8]:
self.covars_encoder

{'cell_types_rand': {'B_cells': 0, 'T_cells': 1, 'macrophages': 2},
 'cat_2': {'A': 0, 'B': 1, 'C': 2, 'D': 3}}

In [9]:
n_genes=adata.n_vars
n_perts=len(self.pert_encoder)
covars_encoder=self.covars_encoder

n_latent = 128
recon_loss = 'nb'
doser_type = 'logsim'
n_hidden_encoder = 512
n_layers_encoder = 3
n_hidden_decoder = 512
n_layers_decoder = 3
n_hidden_doser = 128
n_layers_doser = 2
use_batch_norm_encoder = True
use_layer_norm_encoder = False
use_batch_norm_decoder = True
use_layer_norm_decoder = False
dropout_rate_encoder = 0.1
dropout_rate_decoder = 0.1
variational = False
seed = 434

In [10]:
from scvi.nn import FCLayers
from torch import nn
from scvi.nn import Encoder, DecoderSCVI
from cpa._utils import PerturbationNetwork

class VanillaEncoder(nn.Module):
    def __init__(
            self,
            n_input,
            n_output,
            n_hidden,
            n_layers,
            n_cat_list,
            use_layer_norm=True,
            use_batch_norm=False,
            output_activation: str = 'linear',
            dropout_rate: float = 0.1,
            activation_fn=nn.ReLU,
    ):
        super().__init__()
        self.n_output = n_output
        self.output_activation = output_activation

        self.network = FCLayers(
            n_in=n_input,
            n_out=n_hidden,
            n_cat_list=n_cat_list,
            n_layers=n_layers,
            n_hidden=n_hidden,
            use_layer_norm=use_layer_norm,
            use_batch_norm=use_batch_norm,
            dropout_rate=dropout_rate,
            activation_fn=activation_fn,
        )
        self.z = nn.Linear(n_hidden, n_output)

    def forward(self, inputs, *cat_list):
        if self.output_activation == 'linear':
            z = self.z(self.network(inputs, *cat_list))
        elif self.output_activation == 'relu':
            z = F.relu(self.z(self.network(inputs, *cat_list)))
        else:
            raise ValueError(f'Unknown output activation: {self.output_activation}')
        return z

In [246]:
class Encoder(nn.Module):
    """Build a fully connected encoder that projects input TF activity to a latent space.
    
    Adapted from scVI's `FCLayers`."""

    DEFAULT_HYPER_PARAMS = {'n_latent': 64, 'batch_norm': True, 'layer_norm': False, 'dropout_rate': 0.1,
                            'activation_fn': nn.ReLU # can make as None to have purely linear
               }
    
    def __init__(self, n_features: int, 
                 n_latent: int = 64, 
                 batch_norm: bool = True, 
                 layer_norm: bool = False,
                 dropout_rate: int | float = 0.1,
                 activation_fn: nn.Module | None = nn.ReLU):
        """Initialize encoder.

        Parameters
        ----------
        n_features : int
            the full number of features input to the encoder
        n_latent : int, optional
            dimension (no. of features) of the latent space, by default 64
        batch_norm : bool, optional
            whether to have `BatchNorm` layers or not, by default True
        layer_norm : bool, optional
            whether to have `LayerNorm` layers or not, by default False
        dropout_rate : int | float, optional
            dropout rate to apply to each of the hidden layers, by default 0.1
        activation_fn : nn.Module | None, optional
            non-linear Pytorch activation function, by default nn.ReLU. No activation if set to None
        """
        super().__init__()
        n_in, n_out = n_features, n_latent
        self.encoder = nn.Sequential(
            nn.Linear(in_features = n_in, out_features = n_out, bias = True),
            nn.BatchNorm1d(n_out, momentum=0.01, eps=0.001) if batch_norm else None,
            nn.LayerNorm(n_out, elementwise_affine=False) if layer_norm else None,
            activation_fn() if activation_fn else None,
            nn.Dropout(p=dropout_rate) if (dropout_rate and dropout_rate > 0) else None
        )

    def forward(self, x):
        return self.encoder(x)

In [266]:
import pandas as pd
from typing import List, Any, Dict, Union, Literal
import torch.nn as nn

class TFA(nn.Module):
    """Decompose TF activity into basal effects and covariate-specific effects. 
    
    Adapted from Compositional Perturbation Autoencoder (https://doi.org/10.15252/msb.202211517)."""
    def __init__(self, cat_covariates: pd.DataFrame, 
                 categorical_covariate_keys: List[str],
                 n_latent: int = 64, 
                 cat_max_norm: int | float | None = 1, 
                 recon_loss : Literal['gauss', 'nb'] = 'gauss',
                encoder_hyper_params: Dict[str, Any] = Encoder.DEFAULT_HYPER_PARAMS, 
                device: str = 'cpu'):
        """Initializes model layers.
    
        Parameters
        ----------
        covariates : pd.DataFrame
            metadata with index as sample ids and columns containing various metadata values/mappings
        categorical_covariate_keys : List[str]
            the columns in the dataframe representing categorical/discrete variables
        n_latent: int, optional
            dimension (no. of featuers) of the latent space, by default 64
        cat_max_norm : int | float | None, optional
            passed to `max_norm` argument of nn.Embedding when generating categorical covariate embeddings, by default 1
        recon_loss : Literal['gauss', 'nb'], optional
            Autoencoder loss (either "gauss" or "nb")
            Currently can only handle "guass"
        encoder_hyper_params : Dict[str, Any]
            Key word arguments to pass to `Encoder`. Keys include:
                batch_norm : bool, optional
                    whether to have `BatchNorm` layers or not, by default True
                layer_norm : bool, optional
                    whether to have `LayerNorm` layers or not, by default False
                dropout_rate : int | float, optional
                    dropout rate to apply to each of the hidden layers, by default 0.1
                activation_fn : nn.Module | None,optional
                    non-linear Pytorch activation function, by default nn.ReLU. No activation if set to None
        device : str
            whether to use gpu ("cuda") or cpu ("cpu"), by default "cpu"
        """
        super().__init__()

        self.device = device

        # encoder
        #n_features = ??
        encoder_hyper_params['n_latent'] = n_latent
        self.encoder = Encoder(n_features = n_features, **encoder_hyper_params)

        # categorical embeddings
        # create an embedding for each discrete covariate
        self.map_cat_covariates(cat_covariates, categorical_covariate_keys)
        self.cat_embeddings = nn.ModuleDict(
            {
                covariate_cat: nn.Embedding(num_embeddings = len(covariate_cat_map), embedding_dim = n_latent, 
                                           max_norm = cat_max_norm, norm_type = 2) 
                for covariate_cat, covariate_cat_map in self.cat_mapper.items()}
        )
    
        if recon_loss == 'gauss':
            pass
        else:
            raise ValueError('Currently, TPA can only handle a guassian loss.')

    def map_cat_covariates(self, covariates: pd.DataFrame, categorical_covariate_keys: List[str]):
        """Creates a dictionary mapping each categorical covariate's values to a numerical value (index). 
    
        Parameters
        ----------
        cat_covariates : pd.DataFrame
            metadata with index as sample ids and columns containing various metadata values/mappings
        categorical_covariate_keys : List[str]
            the columns in the dataframe representing categorical/discrete variables
        """

        self.cat_mapper = {}
        for cvk in categorical_covariate_keys:
            if covariates[cvk].dtype.name == 'category' and covariates[cvk].dtype.ordered:
                labels = covariates[cvk].cat.categories
            else:
                labels = sorted(set(covariates[cvk]))
            
            self.cat_mapper[cvk] = {k: idx for idx, k in enumerate(labels)}

In [267]:
hyper_params = {'n_latent': 64, 'cat_max_norm': 1, 'recon_loss': 'gauss'}
encoder_hyper_params = {'batch_norm': True, 'layer_norm': False, 'dropout_rate': 0.1,
                       'activation_fn': nn.ReLU, # can make as None to have purely linear
                       }

self = TFA(cat_covariates = adata.obs.copy(), 
           categorical_covariate_keys = ['cell_types_rand', 'cat_2'],
           n_latent = 64, 
          cat_max_norm = 1, 
          encoder_hyper_params = encoder_hyper_params, 
        device = device)

In [263]:
self.cat_mapper

{'cell_types_rand': {'B_cells': 0, 'T_cells': 1, 'macrophages': 2},
 'cat_2': {'A': 0, 'B': 1, 'C': 2, 'D': 3}}

In [164]:
test = self.cat_embeddings['cell_types_rand']

In [165]:
ct = 'B_cells'
val = torch.tensor(self.cat_mapper['cell_types_rand'][ct], device = self.device, dtype = torch.long)

In [166]:
for name, param in test.named_parameters():
    print(name)

weight


In [167]:
list(test.parameters())[0][0]

tensor([-0.6868,  0.3778, -0.4762,  0.2315,  0.3509, -0.5542, -1.2183,  0.8565,
         0.6826, -0.6228,  0.4919, -1.0190,  0.9590,  0.5226, -1.6493, -0.0959,
        -0.4111, -0.7656,  0.4188,  0.1025, -0.0787,  1.2752, -0.3902, -0.2727,
        -0.9388,  0.6709, -0.9004, -0.6084,  0.8218, -0.3367,  0.6136, -0.8997,
         0.1556,  0.9907, -0.1307,  1.7231,  0.7654,  0.6244, -0.4161, -0.6004,
         2.0849,  0.2839,  0.4826,  0.9614,  0.2215,  1.2258,  0.1981,  0.4752,
         1.8254,  1.1219, -1.0144, -0.4971,  0.3933,  1.1348, -0.5730,  1.3161,
        -0.3314, -0.1108,  1.3930,  0.3396,  1.4079,  0.4800,  0.0503,  0.8768],
       grad_fn=<SelectBackward0>)

In [168]:
self.encoder = VanillaEncoder(
    n_input=n_genes,
    n_output=n_latent,
    n_cat_list=[],
    n_hidden=n_hidden_encoder,
    n_layers=n_layers_encoder,
    use_batch_norm=use_batch_norm_encoder,
    use_layer_norm=use_layer_norm_encoder,
    dropout_rate=dropout_rate_encoder,
    activation_fn=nn.ReLU,
    output_activation='linear',
)

In [61]:


self.encoder = VanillaEncoder(
                n_input=n_genes,
                n_output=n_latent,
                n_cat_list=[],
                n_hidden=n_hidden_encoder,
                n_layers=n_layers_encoder,
                use_batch_norm=use_batch_norm_encoder,
                use_layer_norm=use_layer_norm_encoder,
                dropout_rate=dropout_rate_encoder,
                activation_fn=nn.ReLU,
                output_activation='linear',
            )
self.decoder = Encoder(n_input=n_latent,
                                   n_output=n_genes,
                                   n_layers=n_layers_decoder,
                                   n_hidden=n_hidden_decoder,
                                   dropout_rate=dropout_rate_decoder,
                                   use_batch_norm=use_batch_norm_decoder,
                                   use_layer_norm=use_layer_norm_decoder,
                                   var_activation=None,
                                   )

self.pert_network = PerturbationNetwork(n_perts=n_perts,
                                        n_latent=n_latent,
                                        doser_type=doser_type,
                                        n_hidden=n_hidden_doser,
                                        n_layers=n_layers_doser,
                                        drug_embeddings=drug_embeddings,
                                        )

self.pert_network = PerturbationNetwork(n_perts=n_perts,
                                        n_latent=n_latent,
                                        doser_type=doser_type,
                                        n_hidden=n_hidden_doser,
                                        n_layers=n_layers_doser,
                                        drug_embeddings=drug_embeddings,
                                        )

self.covars_embeddings = nn.ModuleDict(
    {
        key: torch.nn.Embedding(len(unique_covars), n_latent)
        for key, unique_covars in self.covars_encoder.items()
    }
)